# M1.4 · Búsqueda web 🔎

**Lección oficial:** *Web search* · **Notebook original:** `module-1/1.2_web_search.ipynb`

## 🎯 Qué vas a aprender
- Por qué un modelo **no sabe** cosas recientes (*fecha de corte*).
- Cómo darle una tool de **búsqueda web** (Tavily).
- Cómo verificar que el agente **sí buscó** y no se lo inventó.

## 🧸 Explicado sencillo
Un modelo es como un estudiante que leyó una biblioteca enorme… **hasta cierta fecha**, y después lo encerraron en una habitación sin internet 📚🚪. Si le preguntas algo de hoy, adivina con lo último que leyó.

La búsqueda web es **abrirle la ventana** para que mire las noticias.

⚠️ Necesitas `TAVILY_API_KEY` en tu `.env` (gratis en tavily.com).

## 🛠️ Paso 0 · Preparar el notebook

Esta celda hace tres cosas por ti:
1. Encuentra la carpeta principal del repo (donde vive `local_model.py`).
2. Lee tu archivo `.env` (tus llaves y la dirección del servidor) **sin mostrarlas**.
3. Revisa el **semáforo** del servidor Ollama, que es **compartido**:
   - 🟢 libre → adelante.
   - 🟡 hay un modelo cargado → si fuiste tú hace un rato, sigue; si no, espera.
   - 🔴 sin conexión → revisa `OLLAMA_BASE_URL` en tu `.env`.

> Corre las celdas **en orden, de arriba hacia abajo** (Shift + Enter). El número `[n]` a la izquierda te dice en qué orden se ejecutaron.

In [ ]:
# 🟢 ADAPTADO LOCAL: preparar entorno (ver ayuda.py)
from ayuda import preparar, ver, semaforo
preparar()

from local_model import get_model   # se importa DESPUÉS de cargar el .env

## 1 · Sin búsqueda web
Preguntémosle hasta cuándo sabe cosas, y algo actual.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre
# agent = create_agent(model="gpt-5-nano")

# 🟢 ADAPTADO LOCAL
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=get_model())

response = agent.invoke({"messages": [HumanMessage(content="How up to date is your training knowledge?")]})
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="Who is the current president of Colombia?")]})
print(response["messages"][-1].content)

🤔 **¿Te dio una respuesta segura?** Puede ser vieja o incorrecta, y el modelo no te avisa. Por eso necesitamos la ventana 🪟.

## 2 · Crear la tool de búsqueda 🔎
Envolvemos el cliente de Tavily en una tool. Fíjate en la docstring: *"Search the web for information"*. Con eso el modelo entiende cuándo usarla.

`max_results=3` → pedimos solo 3 resultados. Menos texto = el modelo local responde **más rápido** (cada resultado son tokens que hay que leer).

In [ ]:
from typing import Any, Dict
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()          # lee TAVILY_API_KEY del .env

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query, max_results=3)

# Probamos la tool a mano (sin agente)
resultado = web_search.invoke("Who is the current mayor of San Francisco?")
for r in resultado["results"]:
    print("-", r["title"], "→", r["url"])

## 3 · Agente con búsqueda web

In [ ]:
agent = create_agent(
    model=get_model(),
    tools=[web_search],
    system_prompt="Use web_search for any question about current events. Answer briefly and cite the source URL.",
)

response = agent.invoke({"messages": [HumanMessage(content="Who is the current president of Colombia?")]})
ver(response)

## 4 · ¿Buscó de verdad? 🕵️
**Regla de oro:** no confíes en la respuesta, **mira los mensajes**.
- Si hay un 🤖 *"Modelo pide usar la tool → web_search"* y un 🔧 *"Resultado"*, buscó.
- Si solo hay Humano → Modelo, respondió de memoria.

In [ ]:
usó_tool = any(getattr(m, "tool_calls", None) for m in response["messages"])
print("¿El agente buscó en la web?", "✅ Sí" if usó_tool else "❌ No, respondió de memoria")

## 🔭 Extra: LangSmith
Si tienes `LANGSMITH_TRACING=true` en tu `.env`, entra a smith.langchain.com → tu proyecto. Ahí ves cada paso del agente: qué buscó, qué le devolvió Tavily y cuánto tardó. Es como `ver()`, pero con lupa.

## ✍️ Tu turno
1. Pregúntale algo actual que te interese (un partido, un estreno, el clima de tu ciudad).
2. Comprueba con `ver()` si buscó.
3. Reto: quita la frase *"Use web_search…"* del system prompt. ¿Sigue buscando? Si el modelo no busca, es por eso: a veces hay que **decirle** que use la tool.

## ✅ Mini quiz
1. ¿Por qué un modelo no sabe noticias de hoy?
2. ¿Para qué sirve `max_results=3`?
3. ¿Cómo sabes con seguridad si el agente buscó?

<details><summary>Respuestas</summary>

1. Porque aprendió hasta una fecha de corte.
2. Para traer menos texto: menos tokens y respuesta más rápida.
3. Mirando los mensajes: tiene que haber un `tool_calls` y un mensaje de tool.
</details>

## 🧠 Resumen en una línea
**Modelo = estudiante encerrado con libros viejos. Web search = abrirle la ventana. Verifica siempre con los mensajes.**